# TUGAS PRAKTIKUM 5 KECERDASAN KOMPUTASIONAL

| Informasi Mahasiswa | |
| :--- | :--- |
| **Nama** | Catherine Aprilia Harlina |
| **NRP** | 5054251012 |
| **Program Studi** | Rekayasa Kecerdasan Artifisial |
| **Mata Kuliah** | Kecerdasan Komputasional |

---

In [1]:
# Standard setup cell - run before other cells.
import importlib.util
import subprocess
import sys
from pathlib import Path

def ensure_dependencies():
    """Install only packages used by this module."""
    required = {"numpy": "numpy", "matplotlib": "matplotlib", "pandas": "pandas"}
    missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_module_dir(start):
    """Find the shared membership module."""
    for root in [start, *start.parents]:
        for candidate in (root, root / "Fuzzy Logic"):
            if (candidate / "membership.py").exists():
                return candidate
        if (root / ".git").exists():
            break
    return None

ensure_dependencies()
module_dir = find_module_dir(Path.cwd())
if module_dir is None and "google.colab" in sys.modules:
    clone_dir = Path("Modul-Praktikum-KK-RKA-25")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/kcv-if/Modul-Praktikum-KK-RKA-25.git", str(clone_dir)], check=True)
    module_dir = find_module_dir(clone_dir)
if module_dir is None:
    raise RuntimeError("membership.py not found; open this notebook inside the repository")
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from membership import linear_up, linear_down, triangular, trapezoidal

print("Python:", sys.version.split()[0])
print("Membership module:", module_dir / "membership.py")

Python: 3.12.4
Membership module: c:\Users\LEGION\Praktikum-KK\fuzzy-logic\membership.py


# Soal 1 - Pengaturan Kecepatan Kipas Sugeno Orde Nol

## 1. Perhitungan Manual untuk T = 28 C

### a. Derajat Keanggotaan Input T = 28 C
sejuk = linear_down(28, 20, 30) = (30 - 28) / (30 - 20) = 2 / 10 = 0.20
nyaman = triangular(28, 20, 30, 40) = (28 - 20) / (30 - 20) = 8 / 10 = 0.80
panas = linear_up(28, 30, 40) = 0.00

### b. Kekuatan Aktivasi wi dan Evaluasi Konsekuen zi
R1: IF suhu Sejuk THEN z1 = 20 persen -> w1 = 0.20, z1 = 20
R2: IF suhu Nyaman THEN z2 = 50 persen -> w2 = 0.80, z2 = 50
R3: IF suhu Panas THEN z3 = 90 persen -> w3 = 0.00, z3 = 90

### c. Rata Rata Berbobot
sum(wi) = 0.20 + 0.80 + 0.00 = 1.00
sum(wi * zi) = (0.20 * 20) + (0.80 * 50) + (0.00 * 90) = 4.0 + 40.0 + 0 = 44.0
output = 44.0 / 1.00 = 44.00 persen

---

## 2. Perhitungan Manual untuk Pembanding T = 36 C

### a. Derajat Keanggotaan Input T = 36 C
sejuk = linear_down(36, 20, 30) = 0.00
nyaman = triangular(36, 20, 30, 40) = (40 - 36) / (40 - 30) = 4 / 10 = 0.40
panas = linear_up(36, 30, 40) = (36 - 30) / (40 - 30) = 6 / 10 = 0.60

### b. Kekuatan Aktivasi wi dan Evaluasi Konsekuen zi
R1: IF suhu Sejuk THEN z1 = 20 persen -> w1 = 0.00, z1 = 20
R2: IF suhu Nyaman THEN z2 = 50 persen -> w2 = 0.40, z2 = 50
R3: IF suhu Panas THEN z3 = 90 persen -> w3 = 0.60, z3 = 90

### c. Rata Rata Berbobot
sum(wi) = 0.00 + 0.40 + 0.60 = 1.00
sum(wi * zi) = (0.00 * 20) + (0.40 * 50) + (0.60 * 90) = 0 + 20.0 + 54.0 = 74.0
output = 74.0 / 1.00 = 74.00 persen

---

## 3. Tabel Perbandingan dan Analisis

| Suhu | w1 Sejuk | w2 Nyaman | w3 Panas | Output Kipas | Aturan Aktif | Rentang Konsekuen Aktif |
| --- | --- | --- | --- | --- | --- | --- |
| 28 C | 0.20 | 0.80 | 0.00 | 44.00 persen | R1, R2 | 20 sampai 50 |
| 36 C | 0.00 | 0.40 | 0.60 | 74.00 persen | R2, R3 | 50 sampai 90 |

### Pemeriksaan Output dan Analisis Perubahan
pemeriksaan output menunjukkan bahwa output untuk 28 C yaitu 44.00 persen berada di antara konsekuen aturan aktif 20 sampai 50 dan output untuk 36 C yaitu 74.00 persen berada di antara 50 sampai 90
perubahan output terjadi karena saat suhu naik dari 28 C ke 36 C dominasi aturan aktif bergeser dari sejuk nyaman menjadi nyaman panas sehingga kecepatan kipas meningkat dari 44 persen menjadi 74 persen
kurva fungsi keanggotaan output tidak diperlukan pada metode sugeno karena nilai konsekuen setiap aturan sudah berupa konstanta atau persamaan linier bukan himpunan fuzzy sehingga penentuan nilai tegas dapat langsung dihitung menggunakan rata rata berbobot tanpa melalui proses defuzzifikasi berbasis luas area

In [2]:
# panggil/sesuaiin fungsi keanggotaan dari setup modul
def fuzz_temp(t):
    return {'sejuk': linear_down(t, 20, 30), 'nyaman': triangular(t, 20, 30, 40), 'panas': linear_up(t, 30, 40)}
rules_s1 = [{'rule': 'R1', 'in': 'sejuk', 'z': 20.0}, {'rule': 'R2', 'in': 'nyaman', 'z': 50.0}, {'rule': 'R3', 'in': 'panas', 'z': 90.0}]
def eval_s1(t):
    deg = fuzz_temp(t)
    w, z = np.array([deg[r['in']] for r in rules_s1]), np.array([r['z'] for r in rules_s1])
    out = np.sum(w * z) / np.sum(w)
    act = w > 0
    assert z[act].min() <= out <= z[act].max()
    return deg, w, z, out
for temp in [28, 36]:
    deg, w, z, out = eval_s1(temp)
    df = pd.DataFrame({'rule': [r['rule'] for r in rules_s1], 'w_i': w, 'z_i': z, 'w_i*z_i': w * z})
    print(f"suhu {temp}°C")
    display(df.round(4))
    print(f"sum_w: {w.sum():.2f}, sum_wz: {(w*z).sum():.2f}, output: {out:.2f}%\n")

suhu 28°C


,rule,w_i,z_i,w_i*z_i
0,R1,0.2,20.0,4.0
1,R2,0.8,50.0,40.0
2,R3,0.0,90.0,0.0


sum_w: 1.00, sum_wz: 44.00, output: 44.00%

suhu 36°C


,rule,w_i,z_i,w_i*z_i
0,R1,0.0,20.0,0.0
1,R2,0.4,50.0,20.0
2,R3,0.6,90.0,54.0


sum_w: 1.00, sum_wz: 74.00, output: 74.00%



# Soal 2 - durasi pencucian dengan dua input
## 1. perhitungan manual buat input utama (k = 65, b = 6.5 kg)

### a. derajat keanggotaan
* **Kekotoran (k = 65):**
  * rendah = 0.00
  * sedang = (80 - 65) / (80 - 60) = 15 / 20 = 0.75
  * tinggi = (65 - 50) / (80 - 50) = 15 / 30 = 0.50

* **Berat (b = 6.5 kg):**
  * ringan = 0.00
  * sedang = (8 - 6.5) / (8 - 5) = 1.5 / 3 = 0.50
  * berat = (6.5 - 5) / (8 - 5) = 1.5 / 3 = 0.50

### b. kekuatan aktivasi (w_i) & evaluasi konsekun (z_i)
* R5 (sedang, sedang) : w5 = min(0.75, 0.50) = 0.50, z5 = 35
* R6 (sedang, berat)  : w6 = min(0.75, 0.50) = 0.50, z6 = 50
* R8 (tinggi, sedang) : w8 = min(0.50, 0.50) = 0.50, z8 = 50
* R9 (tinggi, berat)  : w9 = min(0.50, 0.50) = 0.50, z9 = 60

### c. rata-rata berbobot
* sum(w_i) = 0.50 + 0.50 + 0.50 + 0.50 = 2.00
* sum(w_i * z_i) = (0.50 * 35) + (0.50 * 50) + (0.50 * 50) + (0.50 * 60) = 97.50
* Output = 97.50 / 2.00 = 48.75 menit

---

## 2. perhitungan manual buat pembanding (k = 35, b = 6.5 kg)

### a. derajat keanggotaan
* **Kekotoran (k = 35):**
  * rendah = (50 - 35) / (50 - 20) = 0.50
  * sedang = (35 - 20) / (40 - 20) = 0.75
  * tinggi = 0.00

* **Berat (b = 6.5 kg):**
  * ringan = 0.00, sedang = 0.50, berat = 0.50

### b. kekuatan aktivasi (w_i) & evaluasi konsekun (z_i)
* R2 (rendah, sedang) : w2 = min(0.50, 0.50) = 0.50, z2 = 20
* R3 (rendah, berat)  : w3 = min(0.50, 0.50) = 0.50, z3 = 30
* R5 (sedang, sedang) : w5 = min(0.75, 0.50) = 0.50, z5 = 35
* R6 (sedang, berat)  : w6 = min(0.75, 0.50) = 0.50, z6 = 50

### c. rata-rata berbobot
* sum(w_i) = 0.50 + 0.50 + 0.50 + 0.50 = 2.00
* sum(w_i * z_i) = (0.50 * 20) + (0.50 * 30) + (0.50 * 35) + (0.50 * 50) = 67.50
* Output = 67.50 / 2.00 = 33.75 menit

## 3. tabel perbandingan dan analisis

| kasus | w1 | w2 | w3 | w4 | w5 | w6 | w7 | w8 | w9 | output (menit) | aturan aktif | rentang konsekuen aktif |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| k=65, b=6.5 | 0.0 | 0.0 | 0.0 | 0.0 | 0.5 | 0.5 | 0.0 | 0.5 | 0.5 | 48.75 | R5, R6, R8, R9 | 35 sampai 60 |
| k=35, b=6.5 | 0.0 | 0.5 | 0.5 | 0.0 | 0.5 | 0.5 | 0.0 | 0.0 | 0.0 | 33.75 | R2, R3, R5, R6 | 20 sampai 50 |

### analisis aturan yang memengaruhi perubahan durasi

pemeriksaan output nunjukin durasi melht penurunan dari 48.75 menit jadi 33.75 menit pas indeks kekotoran turun dri 65 ke 35 padahal beratnya tetep 6.5 kg

aturan yg paling ngaruh sm perubahan ini tuh grgr matinya aturan R8 dan R9 yg tadinya punya bobot zi gede yaitu z8=50 dan z9=60 akibat derajat kekotoran tinggi drop ke 0.00

disaat bersamaan R2 sm R3 mulai aktif membawa nilai konsekuen lebih kecil z2=20 dan z3=30 makanya pas dihitung pke weighted average nilainya langsung kecenderungannya drop drastis

In [3]:
import numpy as np
import pandas as pd
def fuzz_k(k):
    return {
        "rendah": linear_down(k, 20, 50),
        "sedang": trapezoidal(k, 20, 40, 60, 80),
        "tinggi": linear_up(k, 50, 80),
    }
def fuzz_b(b):
    return {
        "ringan": linear_down(b, 2, 5),
        "sedang": triangular(b, 2, 5, 8),
        "berat": linear_up(b, 5, 8),
    }
def weighted_average(w, z):
    return np.sum(w * z) / np.sum(w)
rules_s2 = [
    {"rule": "R1", "k": "rendah", "b": "ringan", "z": 15.0},
    {"rule": "R2", "k": "rendah", "b": "sedang", "z": 20.0},
    {"rule": "R3", "k": "rendah", "b": "berat", "z": 30.0},
    {"rule": "R4", "k": "sedang", "b": "ringan", "z": 20.0},
    {"rule": "R5", "k": "sedang", "b": "sedang", "z": 35.0},
    {"rule": "R6", "k": "sedang", "b": "berat", "z": 50.0},
    {"rule": "R7", "k": "tinggi", "b": "ringan", "z": 30.0},
    {"rule": "R8", "k": "tinggi", "b": "sedang", "z": 50.0},
    {"rule": "R9", "k": "tinggi", "b": "berat", "z": 60.0},
]
def eval_s2(k, b):
    dk, db = fuzz_k(k), fuzz_b(b)
    w = np.array([min(dk[r["k"]], db[r["b"]]) for r in rules_s2])
    z = np.array([r["z"] for r in rules_s2])
    out = weighted_average(w, z)
    return dk, db, w, z, out
for k_val, b_val in [(65, 6.5), (35, 6.5)]:
    dk, db, w, z, out = eval_s2(k_val, b_val)
    df = pd.DataFrame(
        {"rule": [r["rule"] for r in rules_s2], "w_i": w, "z_i": z, "w_i*z_i": w * z}
    )
    print(f"Kekotoran {k_val}, Berat {b_val} kg")
    display(df.round(4))
    print(f"sum_w: {w.sum():.2f}, sum_wz: {(w*z).sum():.2f}, Output: {out:.2f} menit\n")

Kekotoran 65, Berat 6.5 kg


,rule,w_i,z_i,w_i*z_i
0,R1,0.0,15.0,0.0
1,R2,0.0,20.0,0.0
2,R3,0.0,30.0,0.0
3,R4,0.0,20.0,0.0
4,R5,0.5,35.0,17.5
5,R6,0.5,50.0,25.0
6,R7,0.0,30.0,0.0
7,R8,0.5,50.0,25.0
8,R9,0.5,60.0,30.0


sum_w: 2.00, sum_wz: 97.50, Output: 48.75 menit

Kekotoran 35, Berat 6.5 kg


,rule,w_i,z_i,w_i*z_i
0,R1,0.0,15.0,0.0
1,R2,0.5,20.0,10.0
2,R3,0.5,30.0,15.0
3,R4,0.0,20.0,0.0
4,R5,0.5,35.0,17.5
5,R6,0.5,50.0,25.0
6,R7,0.0,30.0,0.0
7,R8,0.0,50.0,0.0
8,R9,0.0,60.0,0.0


sum_w: 2.00, sum_wz: 67.50, Output: 33.75 menit



# Soal 3 - Durasi Penyiraman, Sugeno Orde Satu

hitung manualnya ginnni yaa guyss

### 1. fuzzifikasi input
- kelembapan (h = 45%):
  - Kering = linear_down(45, 30, 60) = (60 - 45) / 30 = 0.5
  - Cukup = trapezoidal(45, 30, 45, 60, 75) = 1.0
  - Basah = linear_up(45, 60, 80) = 0.0

- suhu T = 32°C:
  - Sejuk = linear_down(32, 20, 30) = 0.0
  - Panas = linear_up(32, 25, 35) = (32 - 25) / 10 = 0.7
  - NOT Panas = 1 - 0.7 = 0.3

- suhu T = 27°C:
  - Sejuk = linear_down(27, 20, 30) = (30 - 27) / 10 = 0.3
  - Panas = linear_up(27, 25, 35) = (27 - 25) / 10 = 0.2
  - NOT Panas = 1 - 0.2 = 0.8

---

### 2. evaluasi aturan & output

| Aturan | Anteseden | Rumus Konsekuen (zi) | w (32°C) | z (32°C) | w (27°C) | z (27°C) |
|---|---|---|---|---|---|---|
| R1 | Kering AND Panas | 18 - 0.10h + 0.25T | min(0.5, 0.7) = 0.5 | 21.50 | min(0.5, 0.2) = 0.2 | 20.25 |
| R2 | Kering AND NOT Panas | 12 - 0.05h + 0.10T | min(0.5, 0.3) = 0.3 | 12.95 | min(0.5, 0.8) = 0.5 | 12.45 |
| R3 | Cukup AND Panas | 12 - 0.05h + 0.10T | min(1.0, 0.7) = 0.7 | 12.95 | min(1.0, 0.2) = 0.2 | 12.45 |
| R4 | Cukup AND NOT Panas | 8 - 0.03h + 0.05T | min(1.0, 0.3) = 0.3 | 8.25 | min(1.0, 0.8) = 0.8 | 8.00 |
| R5 | Basah OR Sejuk | 6 - 0.02h + 0.04T | max(0.0, 0.0) = 0.0 | 6.38 | max(0.0, 0.3) = 0.3 | 6.18 |

- kondisi 1 (T = 32°C):
  - total w = 0.5 + 0.3 + 0.7 + 0.3 + 0.0 = 1.8
  - total w * z = (0.5 * 21.5) + (0.3 * 12.95) + (0.7 * 12.95) + (0.3 * 8.25) + 0 = 26.175
  - output = 26.175 / 1.8 = 14.5417 menit

- kondisi 2 (T = 27°C):
  - total w = 0.2 + 0.5 + 0.2 + 0.8 + 0.3 = 2.0
  - total w * z = (0.2 * 20.25) + (0.5 * 12.45) + (0.2 * 12.45) + (0.8 * 8.0) + (0.3 * 6.18) = 21.019
  - output = 21.019 / 2.0 = 10.5095 menit

---

### 3. analisis jawaban

- perbandingan kekuatan R2, R4, dan R5:
  - R2 naik dari 0.3 jadi 0.5
  - R4 naik lumayan jauh dr 0.3 ke 0.8 
  - R5 naik dr 0.0 jd 0.3

- pemeriksaan range output:
  - pas T = 32°C dapet output 14.5417 menit mmg ada di rentang z aturan yg aktif antara 8.25 sampe 21.5
  - trs pas T = 27°C outputnya 10.5095 menit jg masi masuk di rentang z aturan aktifnya dr 6.18 sampe 20.25

- penjelasan perubahan output:
  durasi siramnya jd makin dikit dr 14.54 menit ke 10.51 menit gara2 suhunya drop dr 32°C ke 27°C, pas suhunya lbh adem evaporasi tanahnya gak secepet itu jd gaperlu nyiram lama2 bgt, basically jd makin hemat air jg

- pemeriksaan Sejuk vs NOT Panas:
  nilai Sejuk sm NOT Panas tu gak beneran sama di sepanjang domain T (15 - 40°C), beda bgt secara range transisinya krn Sejuk tu turun dr 20 - 30°C sedangkan NOT Panas (1 - Panas) baru turun pas range 25 - 35°C jd mmg gak bs disamaain gt ajaa

In [4]:
import numpy as np
import pandas as pd
from membership import linear_down, linear_up, trapezoidal
def weighted_average(w, z):
    return sum(wi * zi for wi, zi in zip(w, z)) / sum(w)
def hitung_sugeno_soal3(h, T):
    kering = linear_down(h, 30, 60)
    cukup = trapezoidal(h, 30, 45, 60, 75)
    basah = linear_up(h, 60, 80)
    sejuk = linear_down(T, 20, 30)
    panas = linear_up(T, 25, 35)
    not_panas = 1 - panas
    w = [
        min(kering, panas),        # R1
        min(kering, not_panas),    # R2
        min(cukup, panas),         # R3
        min(cukup, not_panas),     # R4
        max(basah, sejuk)          # R5
    ]
    z = [
        18 - 0.10*h + 0.25*T,      # z1
        12 - 0.05*h + 0.10*T,      # z2
        12 - 0.05*h + 0.10*T,      # z3
        8 - 0.03*h + 0.05*T,       # z4
        6 - 0.02*h + 0.04*T        # z5
    ]
    out = weighted_average(w, z)
    return w, z, out
w_32, z_32, out_32 = hitung_sugeno_soal3(45, 32)
w_27, z_27, out_27 = hitung_sugeno_soal3(45, 27)
df_hasil = pd.DataFrame({
    'Aturan': ['R1', 'R2', 'R3', 'R4', 'R5'],
    'w (32C)': w_32, 'z (32C)': z_32,
    'w (27C)': w_27, 'z (27C)': z_27
})
display(df_hasil.round(4))
print(f"Output T=32°C : {out_32:.4f} menit (Σw = {sum(w_32):.2f}, Σw*z = {sum(wi*zi for wi, zi in zip(w_32, z_32)):.3f})")
print(f"Output T=27°C : {out_27:.4f} menit (Σw = {sum(w_27):.2f}, Σw*z = {sum(wi*zi for wi, zi in zip(w_27, z_27)):.3f})")
T_domain = np.linspace(15, 40, 101)
sejuk_vals = [linear_down(t, 20, 30) for t in T_domain]
not_panas_vals = [1 - linear_up(t, 25, 35) for t in T_domain]
print("\nApakah derajat 'Sejuk' selalu sama dengan 'NOT Panas'?", np.allclose(sejuk_vals, not_panas_vals))

,Aturan,w (32C),z (32C),w (27C),z (27C)
0,R1,0.5,21.50,0.2,20.25
1,R2,0.3,12.95,0.5,12.45
2,R3,0.7,12.95,0.2,12.45
3,R4,0.3,8.25,0.8,8.00
4,R5,0.0,6.38,0.3,6.18


Output T=32°C : 14.5417 menit (Σw = 1.80, Σw*z = 26.175)
Output T=27°C : 10.5095 menit (Σw = 2.00, Σw*z = 21.019)

Apakah derajat 'Sejuk' selalu sama dengan 'NOT Panas'? False
